# synthgen: примеры генерации синтетических данных

Ноутбук показывает, как фреймворк превращает корпус сырых текстов в обучающие данные для 7 задач:

| # | Задача | Что получаем |
|---|---|---|
| 1 | Классификация текстов | метка + мягкие метки (распределение по классам) |
| 2 | Регрессия по тексту | число + среднее/разброс по сэмплам |
| 3 | NER | спаны с символьными смещениями, типами и нормализованными значениями |
| 4 | NER + отношения | сущности + направленные типизированные отношения |
| 5 | Экстрактивный QA | вопросы (в т.ч. без ответа), ответы-спаны, проверка «читателем» |
| 6 | Экстрактивная суммаризация | выжимка из дословных фрагментов, в т.ч. по запросу |
| 7 | Табличные данные | строка таблицы из каждого документа → DataFrame/Parquet |

**Как устроен пайплайн** (одинаково для всех задач):

```
Document ──► схема задачи (задана вручную или придумана LLM, schema="auto")
         ──► текст (mode="label": текст корпуса | mode="generate": новый текст по мотивам корпуса)
         ──► разметка LLM по JSON-схеме ─► программная валидация ─► самоисправление при ошибках
         ──► n_samples независимых разметок ─► мягкие метки / уверенность
         ──► (опц.) LLM-судья ─► (опц.) рассуждение постфактум ─► Record (канонический JSONL)
         ──► экспорт: Jev-подобный JSONL, chat/SFT с <think>, BIO/CoNLL, SQuAD, Parquet
```

## 0. Установка и подключение к LLM

Ноутбук рассчитан на ядро **Python (synthgen)**: это отдельное виртуальное окружение с установленным пакетом.
Если ноутбук открыт в другом ядре (например, в базовой Anaconda), выберите *Kernel → Change kernel → Python (synthgen)*.

```bash
py -3.12 -m venv .venv
.venv\Scripts\python -m pip install -e ".[all]"
.venv\Scripts\python -m ipykernel install --user --name synthgen --display-name "Python (synthgen)"
```

Настройки LLM читаются из `.env` в корне репозитория (шаблон: `.env.example`). Используется OpenAI-совместимый эндпоинт Ollama.

In [1]:
import json, logging
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

import synthgen as sg
from synthgen import export
from synthgen.viz import highlight

logging.basicConfig(level=logging.WARNING)
pd.set_option("display.max_colwidth", 120)

# Корень репозитория: там лежат data/ и .env
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
llm = sg.LLMClient(sg.LLMConfig.from_env(dotenv_path=ROOT / ".env"))
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)
print("synthgen", sg.__version__, "| pandas", pd.__version__)
print(llm.config.base_url, "|", llm.config.model, "| embeddings:", llm.config.embed_model)

synthgen 0.1.0 | pandas 3.0.6
http://192.168.1.12:11434/v1 | qwen3:30b | embeddings: paraphrase-multilingual


In [2]:
# Быстрая проверка: структурированный ответ по произвольной Pydantic-модели
from pydantic import BaseModel

class Ping(BaseModel):
    answer: str
    lang: str

await llm.structured([{"role": "user", "content": "Скажи 'привет' и укажи код языка ответа"}], Ping)

Ping(answer='привет', lang='ru')

### Ключ API и отдельный сервер эмбеддингов

**Ключ к удалённому серверу** передаётся в заголовке `Authorization: Bearer <ключ>`. Задать его можно тремя способами,
по убыванию предпочтительности:

1. В файле `.env`: `SYNTHGEN_API_KEY=sk-...`. Файл внесён в `.gitignore`, ключ не попадёт в репозиторий.
2. Переменной окружения `SYNTHGEN_API_KEY`. Она важнее значения из `.env`.
3. В коде: `sg.LLMConfig(api_key=...)` или `sg.LLMConfig.from_env(api_key=os.environ["MY_SECRET"])`.

**Эмбеддинги можно брать с другого сервера.** Они нужны для `select_diverse` и `deduplicate`. Параметры задаются отдельно от чат-модели:

| Параметр `LLMConfig` | Переменная `.env` | Если не задан |
|---|---|---|
| `embed_base_url` | `SYNTHGEN_EMBED_BASE_URL` | используется `base_url` |
| `embed_api_key` | `SYNTHGEN_EMBED_API_KEY` | используется `api_key` |
| `embed_model` | `SYNTHGEN_EMBED_MODEL` | эмбеддинги недоступны |

Если сервер эмбеддингов отдельный, у него свой лимит `max_concurrency`, и он не занимает слоты чат-модели.
Подойдёт любой OpenAI-совместимый `/v1/embeddings`: Ollama, vLLM, TEI, OpenAI.

In [3]:
# Какие серверы использует клиент из .env
cfg = llm.config
print("LLM:        ", cfg.base_url, "| модель:", cfg.model, "| ключ задан:", bool(cfg.api_key))
print("Эмбеддинги: ", cfg.embed_base_url or cfg.base_url, "| модель:", cfg.embed_model,
      "| отдельный сервер:", cfg.separate_embed_server)

LLM:         http://192.168.1.12:11434/v1 | модель: qwen3:30b | ключ задан: True
Эмбеддинги:  http://192.168.1.12:11434/v1 | модель: paraphrase-multilingual | отдельный сервер: False


In [4]:
# Пример: чат-модель на удалённом сервере из .env, эмбеддинги на другом сервере (здесь локальная Ollama).
# Значения переопределяют .env только для этого клиента. Ключи лучше хранить в .env, в коде они для наглядности.
split_llm = sg.LLMClient(sg.LLMConfig.from_env(
    dotenv_path=ROOT / ".env",
    embed_base_url="http://localhost:11434/v1",
    embed_api_key="ollama",
    embed_model="mxbai-embed-large",
))
try:
    vec = await split_llm.embed(["привет мир", "hello world"])
    print("чат:        ", split_llm.config.base_url)
    print("эмбеддинги: ", split_llm.config.embed_base_url, "| векторы:", vec.shape)
except Exception as e:
    print(f"Сервер эмбеддингов недоступен ({type(e).__name__}). Поднимите Ollama локально "
          f"(ollama pull mxbai-embed-large) или укажите свой embed_base_url.")

чат:         http://192.168.1.12:11434/v1
эмбеддинги:  http://localhost:11434/v1 | векторы: (2, 1024)


## 1. Корпус

На вход подаётся любой `Iterable[Document]`. В комплекте адаптеры `read_jsonl`, `read_txt_dir`, `from_texts`
и нарезка длинных документов на чанки по границам предложений (`chunk_documents`).
Демо-корпус — 15 коротких текстов на русском и английском: отзывы о банках, новости, объявления о квартирах.

In [5]:
docs = list(sg.chunk_documents(sg.read_jsonl(ROOT / "data" / "sample_corpus.jsonl"), max_chars=1500, min_chars=100))
by_topic = {t: [d for d in docs if d.meta["topic"] == t] for t in ("reviews", "news", "listings")}

pd.DataFrame([{"id": d.id, "topic": d.meta["topic"], "lang": sg.detect_lang(d.text), "text": d.text[:110] + "…"} for d in docs])

,id,topic,lang,text
0,review-1,reviews,ru,"Открыл вклад в Альфа-Банке в отделении на Тверской. Менеджер Ирина всё объяснила за 15 минут, ставка 16% годов…"
1,review-2,reviews,ru,"Третью неделю не могу вернуть 12 400 рублей, списанных дважды за одну покупку. В чате поддержки Т-Банка отвеча…"
2,review-3,reviews,ru,"Подскажите, можно ли в Сбербанке перевыпустить карту «Мир» без визита в отделение? Срок действия заканчивается…"
3,review-4,reviews,ru,"Кредитную карту ВТБ одобрили за день, лимит 150 тысяч. Но при оформлении навязали страховку за 2 990 рублей в …"
4,news-1,news,ru,Компания «Яндекс» объявила 14 мая 2025 года о запуске беспилотных грузовиков на трассе М-11 между Москвой и Са…
5,news-2,news,ru,Правительство Татарстана и корпорация «КАМАЗ» подписали соглашение о строительстве завода аккумуляторов в Набе…
6,news-3,news,ru,Центральный банк России 25 июля сохранил ключевую ставку на уровне 18%. Глава регулятора Эльвира Набиуллина от…
7,news-4,news,ru,Суд в Екатеринбурге признал банкротом девелопера «Атомстройкомплекс». Долги компании перед банками превышают 9…
8,flat-1,listings,ru,"Продаётся двухкомнатная квартира 54 м² в Казани, Советский район, улица Академика Губкина. Седьмой этаж 12-эта…"
9,flat-2,listings,ru,"Сдам студию 26 м² в Москве, район Митино, 5 минут пешком до метро. 3 этаж панельного дома. Без ремонта, но ест…"


Для больших корпусов важно размечать не первые N текстов, а покрывающие разные темы. `select_diverse`
кластеризует эмбеддинги (эндпоинт `/embeddings`) и берёт по представителю из каждого кластера.

In [6]:
emb = await llm.embed([d.text for d in docs])
diverse = sg.select_diverse(docs, emb, k=5)
[(d.id, d.meta["topic"]) for d in diverse]

[('news-2', 'news'),
 ('review-4', 'reviews'),
 ('flat-4', 'listings'),
 ('review-2', 'reviews'),
 ('flat-2', 'listings')]

## 2. Классификация текстов

### 2.1 Разметка текстов корпуса (`mode="label"`) с мягкими метками

`n_samples=3` — три независимые разметки с температурой 0.7. Итоговая метка — большинство голосов,
в `confidence.label_probs` — распределение, пригодное для обучения калиброванной модели (как у Jev).

In [7]:
intent_schema = sg.ClassificationSchema(
    name="bank_review_intent",
    instruction="Определи основную цель обращения клиента банка.",
    labels=[
        sg.LabelDef(name="complaint", description="жалоба на проблему, требование исправить ситуацию"),
        sg.LabelDef(name="praise", description="положительный отзыв, благодарность"),
        sg.LabelDef(name="question", description="вопрос или просьба о консультации"),
        sg.LabelDef(name="mixed", description="смешанный отзыв: есть и плюсы, и минусы"),
    ],
)

gen = sg.Generator(llm, "text_classification", intent_schema, n_samples=3)
cls_ds = await gen.run(by_topic["reviews"])

pd.DataFrame([{"text": r.text[:90] + "…", "label": r.answer["label"], "probs": r.confidence["label_probs"]} for r in cls_ds])

text_classification:   0%|          | 0/5 [00:00<?, ?it/s]

,text,label,probs
0,Открыл вклад в Альфа-Банке в отделении на Тверской. Менеджер Ирина всё объяснила за 15 мин…,praise,"{'complaint': 0.0, 'praise': 1.0, 'question': 0.0, 'mixed': 0.0}"
1,"Третью неделю не могу вернуть 12 400 рублей, списанных дважды за одну покупку. В чате подд…",complaint,"{'complaint': 1.0, 'praise': 0.0, 'question': 0.0, 'mixed': 0.0}"
2,"Подскажите, можно ли в Сбербанке перевыпустить карту «Мир» без визита в отделение? Срок де…",question,"{'complaint': 0.0, 'praise': 0.0, 'question': 1.0, 'mixed': 0.0}"
3,"Кредитную карту ВТБ одобрили за день, лимит 150 тысяч. Но при оформлении навязали страховк…",mixed,"{'complaint': 0.0, 'praise': 0.0, 'question': 0.0, 'mixed': 1.0}"
4,I've been with Chase for eight years and this is the first time I'm seriously considering …,complaint,"{'complaint': 1.0, 'praise': 0.0, 'question': 0.0, 'mixed': 0.0}"


### 2.2 Генерация новых текстов с балансом классов (`mode="generate"`)

Тексты корпуса используются как образцы домена и стиля. Целевые классы раздаются по кругу,
каждый новый текст **независимо переразмечается**, и если метка не совпала с целевой — пример отбрасывается
(round-trip фильтр). Если какой-то класс недобрал, запускается догенерация (`rebalance_rounds`).

In [8]:
gen = sg.Generator(llm, "text_classification", intent_schema, mode="generate", per_document=2, n_samples=3)
gen_ds = await gen.run(by_topic["reviews"])

print("статистика:", dict(gen.stats))
print("распределение классов:", dict(sg.label_distribution(gen_ds)))
pd.DataFrame([{"target": r.quality["target"], "label": r.answer["label"], "text": r.text[:140] + "…"} for r in gen_ds])

text_classification:   0%|          | 0/10 [00:00<?, ?it/s]

rebalance 1:   0%|          | 0/3 [00:00<?, ?it/s]

rebalance 2:   0%|          | 0/1 [00:00<?, ?it/s]

статистика: {'resumed': 0, 'rejected_target': 2, 'ok': 12}
распределение классов: {'complaint': 3, 'praise': 3, 'mixed': 3, 'question': 3}


,target,label,text
0,complaint,complaint,"Вклад Сбербанка, сч. №456789, открыт 10.03. Проценты за март (4500 руб.) должны были начислиться 5-го, но на счету 3..."
1,praise,praise,"Открыл депозит в Тинькофф Банке онлайн. Менеджер Анна быстро оформила, ставка 18% годовых, как в обещанном пакете. П..."
2,mixed,mixed,"Кредит одобрили за 10 минут — это удобно. Но при досрочном погашении списали 750 рублей за скрытую комиссию, хотя в ..."
3,complaint,complaint,"10 октября в 18:45 со счета 5555 6666 7777 произошло неправильное списание 3000 рублей за услугу, которую я не заказ..."
4,praise,praise,"Спасибо Сбербанку за оперативное решение и срочное обслуживание! Вчера попросил заменить утерянную карту «Мир», и уж..."
5,question,question,Могу ли я увеличить лимит по дебетовой карте в Тинькофф с 25 тыс. до 50 тыс. рублей? Нужны ли дополнительные докумен...
6,mixed,mixed,"Кредит в Сбербанке одобрили за 2 часа, лимит 250 тысяч. При оформлении навязали страховку за 2 500 рублей в месяц, о..."
7,complaint,complaint,"Dear First National, My rent payment to 'Maple Apartments' failed to process three business days ago despite correct..."
8,praise,praise,"Sarah Chen here from First National Bank's downtown branch. My mother needed urgent medical funds yesterday, and whe..."
9,mixed,mixed,"Открыл вклад в Тинькофф в отделении на Ленина, менеджер Анна быстро оформила. Ставка 12%, но приложение зависает при..."


## 3. Регрессия по тексту + автоматическое придумывание схем

Здесь схему задаёт не человек: `propose_schemas` показывает LLM выборку корпуса и просит придумать
несколько разных задач регрессии. Это основной механизм разнообразия задач для модели «в духе Jev».

In [9]:
reg = sg.get_task("text_regression")
# Схемы придумываются по отзывам и применяются к отзывам: схема должна подходить к текстам, которые по ней размечаются
reg_schemas = await reg.propose_schemas(llm, [d.text for d in by_topic["reviews"]], k=3, lang="ru")
for s in reg_schemas:
    print(f"• {s.name}: {s.target} [{s.min_value}..{s.max_value}] — {s.scale_description}")

• client_satisfaction_score: удовлетворённость [0.0..10.0] — 0 — крайне недоволен, 5 — нейтрально, 10 — очень доволен
• service_response_time: время ответа [0.0..10.0] — 0 — проблема не решена, 5 — среднее время, 10 — мгновенно решено
• issue_complexity: сложность [0.0..10.0] — 0 — простая проблема, 5 — средняя сложность, 10 — сложная проблема


In [10]:
# Каждый отзыв размечается по каждой схеме; n_samples=3 даёт среднее и разброс оценки
reg_ds = sg.Dataset()
for schema in reg_schemas:
    reg_ds += await sg.Generator(llm, "text_regression", schema, n_samples=3).run(by_topic["reviews"])

pd.DataFrame([{
    "schema": r.task_schema["target"], "value": r.answer["value"],
    "std": round(r.confidence["std"], 2), "samples": r.confidence["samples"], "text": r.text[:80] + "…",
} for r in reg_ds])

text_regression:   0%|          | 0/5 [00:00<?, ?it/s]

text_regression:   0%|          | 0/5 [00:00<?, ?it/s]

text_regression:   0%|          | 0/5 [00:00<?, ?it/s]

,schema,value,std,samples,text
0,удовлетворённость,9.5,0.24,"[9.0, 9.5, 9.5]",Открыл вклад в Альфа-Банке в отделении на Тверской. Менеджер Ирина всё объяснила…
1,удовлетворённость,1.5,0.14,"[1.2, 1.5, 1.5]","Третью неделю не могу вернуть 12 400 рублей, списанных дважды за одну покупку. В…"
2,удовлетворённость,5.0,0.00,"[5.0, 5.0, 5.0]","Подскажите, можно ли в Сбербанке перевыпустить карту «Мир» без визита в отделени…"
3,удовлетворённость,4.5,0.94,"[4.5, 4.5, 6.5]","Кредитную карту ВТБ одобрили за день, лимит 150 тысяч. Но при оформлении навязал…"
4,удовлетворённость,1.5,0.24,"[1.5, 1.5, 2.0]",I've been with Chase for eight years and this is the first time I'm seriously co…
5,время ответа,10.0,0.00,"[10.0, 10.0, 10.0]",Открыл вклад в Альфа-Банке в отделении на Тверской. Менеджер Ирина всё объяснила…
6,время ответа,1.5,0.47,"[1.5, 1.5, 2.5]","Третью неделю не могу вернуть 12 400 рублей, списанных дважды за одну покупку. В…"
7,время ответа,0.0,0.00,"[0.0, 0.0, 0.0]","Подскажите, можно ли в Сбербанке перевыпустить карту «Мир» без визита в отделени…"
8,время ответа,3.0,0.94,"[3.0, 3.0, 5.0]","Кредитную карту ВТБ одобрили за день, лимит 150 тысяч. Но при оформлении навязал…"
9,время ответа,1.0,0.00,"[1.0, 1.0, 1.0]",I've been with Chase for eight years and this is the first time I'm seriously co…


## 4. Выделение именованных сущностей

LLM возвращает **фрагмент текста**, а смещения находит фреймворк: LLM плохо считает символы.
Если фрагмента нет в тексте, модель получает ошибку и исправляет ответ. Для русского языка поиск
учитывает словоформы («Набережные Челны» → «Набережных Челнах»). `with_attributes=True` добавляет
нормализованное значение: дату в ISO, сумму числом, имя в именительном падеже.

In [11]:
ner_schema = sg.NERSchema(
    name="news_ner",
    instruction="Выдели именованные сущности.",
    entity_types=[
        sg.LabelDef(name="PER", description="люди"),
        sg.LabelDef(name="ORG", description="компании, ведомства, организации"),
        sg.LabelDef(name="LOC", description="страны, города, регионы, адреса, трассы"),
        sg.LabelDef(name="DATE", description="даты и годы"),
        sg.LabelDef(name="MONEY", description="денежные суммы с валютой"),
    ],
    with_attributes=True,
)
gen = sg.Generator(llm, "ner", ner_schema, n_samples=3)
ner_ds = await gen.run(by_topic["news"])

for r in ner_ds[:3]:
    display(HTML(highlight(r.text, r.answer["entities"])))

ner:   0%|          | 0/6 [00:00<?, ?it/s]

In [12]:
# Сущности с нормализованными значениями и долей голосов из 3 сэмплов
pd.DataFrame([{k: e.get(k) for k in ("text", "type", "value", "confidence")} for e in ner_ds[0].answer["entities"]])

,text,type,value,confidence
0,14 мая 2025 года,DATE,2025-05-14,1.000
1,М-11,LOC,М-11,1.000
2,Москвой,LOC,Москва,1.000
3,Санкт-Петербургом,LOC,Санкт-Петербург,1.000
4,Дмитрия Полищука,PER,Дмитрий Полищук,0.667
5,20 машин,MONEY,20,0.667
6,"3,5 млрд рублей",MONEY,3500000000 руб,1.000


In [13]:
# Экспорт в BIO/CoNLL для обучения token-classification моделей (BERT и т.п.)
export.to_conll(ner_ds, OUT / "ner.conll")
print("\n".join(f"{t}\t{tag}" for t, tag in export.to_bio(ner_ds[0])[:25]))

Компания	O
«	O
Яндекс	O
»	O
объявила	O
14	B-DATE
мая	I-DATE
2025	I-DATE
года	I-DATE
о	O
запуске	O
беспилотных	O
грузовиков	O
на	O
трассе	O
М	B-LOC
-	I-LOC
11	I-LOC
между	O
Москвой	B-LOC
и	O
Санкт	B-LOC
-	I-LOC
Петербургом	I-LOC
.	O


## 5. Сущности и отношения между ними

Схема задаёт типы отношений и допустимые типы их аргументов. Валидатор проверяет, что `head` и `tail`
ссылаются на выделенные сущности нужных типов; нарушение возвращается модели на исправление.

In [14]:
re_schema = sg.RelationSchema(
    name="business_relations",
    instruction="Выдели участников деловых событий и связи между ними.",
    entity_types=[
        sg.LabelDef(name="PER", description="люди"),
        sg.LabelDef(name="ORG", description="компании и организации"),
        sg.LabelDef(name="LOC", description="географические объекты"),
    ],
    relation_types=[
        sg.RelationDef(name="works_for", description="человек занимает должность в организации", head_types=["PER"], tail_types=["ORG"]),
        sg.RelationDef(name="located_in", description="объект организации расположен или строится в месте", head_types=["ORG"], tail_types=["LOC"]),
        sg.RelationDef(name="partner_of", description="две организации ДРУГ С ДРУГОМ заключили соглашение или сделку (не просто упомянуты рядом)", head_types=["ORG"], tail_types=["ORG"]),
        sg.RelationDef(name="creditor_of", description="организация — кредитор другой организации", head_types=["ORG"], tail_types=["ORG"]),
    ],
)
gen = sg.Generator(llm, "relation_extraction", re_schema, n_samples=3)
re_ds = await gen.run(by_topic["news"])

rows = []
for r in re_ds:
    for rel in r.answer["relations"]:
        rows.append({"doc": r.doc_id, "head": rel["head_text"], "relation": rel["type"], "tail": rel["tail_text"], "conf": rel["confidence"]})
pd.DataFrame(rows)

relation_extraction:   0%|          | 0/6 [00:00<?, ?it/s]

,doc,head,relation,tail,conf
0,news-2,Правительство Татарстана,partner_of,КАМАЗ,1.000
1,news-3,Сбер,creditor_of,Центральный банк России,1.000
2,news-3,Эльвира Набиуллина,works_for,Центральный банк России,1.000
3,news-4,Газпромбанк,creditor_of,Атомстройкомплекс,0.667
4,en-news-1,Deutsche Bank,creditor_of,TenneT,0.667
5,en-news-1,Siemens Energy,partner_of,TenneT,1.000


## 6. Экстрактивный ответ на вопрос

Генератор пишет вопросы, часть из которых **не имеет ответа в тексте** (как в SQuAD 2.0), а ответ может
состоять из нескольких фрагментов. При `n_samples > 1` каждый вопрос независимо проверяется «читателем»:
он отвечает на вопрос `n` раз, а доля совпавших с эталоном ответов становится уверенностью.
Вопросы, на которых читатель не согласен с генератором, отбрасываются. Так отсеиваются ошибки генератора,
например «вопрос без ответа», на который в тексте ответ всё-таки есть.

In [15]:
qa_schema = sg.QASchema(name="facts_qa", instruction="Составь вопросы по фактам из текста.", n_questions=4, unanswerable_ratio=0.25)
gen = sg.Generator(llm, "extractive_qa", qa_schema, n_samples=3)
qa_ds = await gen.run(by_topic["news"] + by_topic["reviews"][:2])

rows = []
for r in qa_ds:
    for qa in r.answer["qas"]:
        rows.append({"doc": r.doc_id, "question": qa["question"], "answer": " | ".join(a["text"] for a in qa["answers"]) or "∅ (нет ответа)",
                     "confidence": qa["confidence"]})
print("отброшено при проверке:", sum(r.confidence["dropped"] for r in qa_ds))
pd.DataFrame(rows)

extractive_qa:   0%|          | 0/8 [00:00<?, ?it/s]

отброшено при проверке: 6


,doc,question,answer,confidence
0,news-4,Какой банк является крупнейшим кредитором компании «Атомстройкомплекс»?,Газпромбанк,1.000
1,news-4,Сколько дольщиков жилого комплекса «Олимп» рискуют не получить квартиры в срок?,800,1.000
2,news-4,"Как называется жилой комплекс, о котором заявил представитель инициативной группы?",«Олимп»,1.000
3,news-1,Когда компания «Яндекс» объявила о запуске беспилотных грузовиков на трассе М-11?,14 мая 2025 года,1.000
4,news-1,Какое количество машин выйдет на маршрут в первый год?,20 машин,1.000
5,news-1,Какую сумму составят инвестиции в проект?,"3,5 млрд рублей",1.000
6,news-3,Что произошло с ключевой ставкой Центрального банка России 25 июля?,сохранил ключевую ставку на уровне 18%,1.000
7,news-3,Какую оценку инфляции дала Эльвира Набиуллина?,инфляция замедляется медленнее прогноза,1.000
8,news-3,Какой прогноз аналитиков «Сбера» относительно снижения ставки?,снижения ставки не раньше октября,1.000
9,news-3,Какой уровень кредитования был отмечен Эльвира Набиуллина?,∅ (нет ответа),1.000


In [16]:
squad = export.to_squad(qa_ds)
(OUT / "qa_squad.json").write_text(json.dumps(squad, ensure_ascii=False, indent=1), encoding="utf-8")
squad["data"][0]["paragraphs"][0]["qas"][:2]

[{'id': 'f2d32018be9dfdd9-0',
  'question': 'Какой банк является крупнейшим кредитором компании «Атомстройкомплекс»?',
  'is_impossible': False,
  'answers': [{'text': 'Газпромбанк', 'answer_start': 146}]},
 {'id': 'f2d32018be9dfdd9-1',
  'question': 'Сколько дольщиков жилого комплекса «Олимп» рискуют не получить квартиры в срок?',
  'is_impossible': False,
  'answers': [{'text': '800', 'answer_start': 165}]}]

## 7. Экстрактивная суммаризация

Выжимка состоит из дословных фрагментов (не обязательно целых предложений). С `query` получается
суммаризация по запросу. При нескольких сэмплах выбирается медоида, то есть сэмпл, больше всего похожий на остальные,
а средняя похожесть становится уверенностью.

In [17]:
risk_schema = sg.SummarizationSchema(
    name="risks_summary", instruction="Сделай выжимку для риск-аналитика.", query="риски, проблемы и негативные последствия", max_spans=3,
)
gen = sg.Generator(llm, "extractive_summarization", risk_schema, n_samples=3)
sum_ds = await gen.run(by_topic["news"])

for r in sum_ds:
    print(f"{r.doc_id}: согласие сэмплов = {r.confidence.get('agreement')}")
    display(HTML(highlight(r.text, r.answer["spans"], label_key=None)))

extractive_summarization:   0%|          | 0/6 [00:00<?, ?it/s]

news-1: согласие сэмплов = 1.0


news-3: согласие сэмплов = 1.0


en-news-1: согласие сэмплов = 1.0


news-2: согласие сэмплов = 0.844


en-news-2: согласие сэмплов = 0.803


news-4: согласие сэмплов = 1.0


## 8. Табличные данные

Каждый документ превращается в строку таблицы по схеме колонок. Модель ставит `null`, если значения нет в тексте,
а не выдумывает его. `export.to_table` приводит типы (Int64, category, datetime), результат готов для CatBoost или sklearn.

Чтобы получить больше строк, чем документов, тот же генератор запускается в режиме `generate`: он пишет новые объявления
по мотивам реальных и извлекает из них строки.

In [18]:
flat_schema = sg.TabularSchema(
    name="flat_listings",
    instruction="Извлеки параметры квартиры из объявления.",
    columns=[
        sg.ColumnDef(name="city", dtype="str", description="город"),
        sg.ColumnDef(name="deal", dtype="category", description="тип сделки", categories=["sale", "rent"]),
        sg.ColumnDef(name="rooms", dtype="int", description="число комнат, студия = 0"),
        sg.ColumnDef(name="area_m2", dtype="float", description="общая площадь", unit="м²"),
        sg.ColumnDef(name="floor", dtype="int", description="этаж квартиры"),
        sg.ColumnDef(name="floors_total", dtype="int", description="этажность дома"),
        sg.ColumnDef(name="build_year", dtype="int", description="год постройки или сдачи дома"),
        sg.ColumnDef(name="wall", dtype="category", description="материал стен", categories=["brick", "panel", "monolith", "other"]),
        sg.ColumnDef(name="renovation", dtype="category", description="состояние", categories=["none", "cosmetic", "good", "designer"]),
        sg.ColumnDef(name="price_rub", dtype="float", description="цена продажи или аренды в месяц, в рублях", unit="руб"),
    ],
    target="price_rub",
)

real = await sg.Generator(llm, "tabular", flat_schema, n_samples=3).run(by_topic["listings"])
synthetic = await sg.Generator(llm, "tabular", flat_schema, mode="generate", per_document=3).run(by_topic["listings"])

table = export.to_table(real + synthetic)
table.insert(1, "source", [r.source for r in real + synthetic])
table.to_parquet(OUT / "flats.parquet")
display(table)
table.dtypes.to_frame("dtype").T

tabular:   0%|          | 0/4 [00:00<?, ?it/s]

tabular:   0%|          | 0/12 [00:00<?, ?it/s]

,doc_id,source,city,deal,rooms,area_m2,floor,floors_total,build_year,wall,renovation,price_rub
0,flat-1,corpus,Казань,sale,2,54.0,7,12,2015,brick,cosmetic,9200000.0
1,flat-2,corpus,Москва,rent,0,26.0,3,<NA>,<NA>,panel,none,45000.0
2,flat-3,corpus,Новосибирск,sale,3,78.0,2,9,2021,monolith,designer,11800000.0
3,flat-4,corpus,Санкт-Петербург,sale,1,38.0,14,25,2019,panel,cosmetic,6100000.0
4,flat-1,generated,Новосибирск,sale,3,72.0,5,10,2010,panel,cosmetic,6800000.0
5,flat-2,generated,Санкт-Петербург,rent,1,32.0,7,9,1985,brick,cosmetic,52000.0
6,flat-1,generated,Екатеринбург,sale,1,42.0,3,10,2016,panel,cosmetic,6300000.0
7,flat-1,generated,Новосибирск,sale,3,78.0,5,9,2010,panel,cosmetic,6500000.0
8,flat-2,generated,Санкт-Петербург,rent,0,32.0,7,9,1995,brick,cosmetic,38000.0
9,flat-2,generated,Санкт-Петербург,rent,1,32.0,7,9,2005,brick,cosmetic,35000.0


,doc_id,source,city,deal,rooms,area_m2,floor,floors_total,build_year,wall,renovation,price_rub
dtype,str,str,str,category,Int64,float64,Int64,Int64,Int64,category,category,float64


In [19]:
# Согласие сэмплов по колонкам: низкое значение означает спорное извлечение
export.to_table(real, with_confidence=True).filter(like="__conf")

,city__conf,deal__conf,rooms__conf,area_m2__conf,floor__conf,floors_total__conf,build_year__conf,wall__conf,renovation__conf,price_rub__conf
0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


## 9. Данные для модели в духе Jev: много разных схем

`schema="auto"`: для **каждого документа** LLM придумывает свою задачу, затем размечает по ней.
Получается набор `(текст, схема выхода, типизированный ответ, вероятности)` с большим разнообразием схем,
то есть обучающие данные для модели, которая отвечает по произвольной схеме, заданной в запросе.

In [20]:
gen = sg.Generator(llm, "text_classification", "auto", n_samples=3)
auto_ds = await gen.run(diverse)

for r in auto_ds:
    print(f"[{r.doc_id}] {r.task_schema['instruction']}")
    print("   классы:", [l["name"] for l in r.task_schema["labels"]], "→", r.answer["label"], r.confidence["label_probs"])

text_classification:   0%|          | 0/5 [00:00<?, ?it/s]

[flat-4] Определите тип квартиры по количеству комнат и площади, чтобы классифицировать объявления на рынке недвижимости.
   классы: ['one_room', 'two_room', 'three_room', 'studio'] → one_room {'one_room': 1.0, 'two_room': 0.0, 'three_room': 0.0, 'studio': 0.0}
[news-2] Определите, описывает ли текст расширение компании или инициативу, связанную с расширением бизнеса, и укажите основную направленность расширения
   классы: ['plant_expansion', 'new_products', 'partnerships', 'market_expansion'] → plant_expansion {'plant_expansion': 1.0, 'new_products': 0.0, 'partnerships': 0.0, 'market_expansion': 0.0}
[review-2] Определите тип проблемы клиента с обслуживанием в банке по его описанию, основываясь на ключевых признаках текста.
   классы: ['double_charge', 'long_wait_time', 'slow_response_chat', 'refund_issue'] → double_charge {'double_charge': 1.0, 'long_wait_time': 0.0, 'slow_response_chat': 0.0, 'refund_issue': 0.0}
[review-4] Определите основной опыт клиента с кредитной картой ВТБ, оп

In [21]:
structured = export.to_structured(auto_ds)
export.write_jsonl(structured, OUT / "jev_style.jsonl")
print(json.dumps(structured[0], ensure_ascii=False, indent=1)[:1500])

{
 "id": "b4918c9c52788636",
 "task": "text_classification",
 "instruction": "Определите тип квартиры по количеству комнат и площади, чтобы классифицировать объявления на рынке недвижимости.\nВыбери ровно один класс.\nКлассы:\n- one_room: Квартира с одним жилым помещением, площадью до 50 м².\n- two_room: Квартира с двумя жилыми помещениями, площадью до 70 м².\n- three_room: Квартира с тремя жилыми помещениями, площадью до 90 м².\n- studio: Студия, не имеющая разделённых комнат на жилые зоны, площадью до 40 м².",
 "input": "Однокомнатная квартира 38 м² в Санкт-Петербурге у метро «Девяткино», 14 этаж из 25. Дом сдан в 2019 году, панель. Требуется косметический ремонт. Стоимость 6,1 млн рублей. Подходит под семейную ипотеку.",
 "output_schema": {
  "properties": {
   "label": {
    "enum": [
     "one_room",
     "two_room",
     "three_room",
     "studio"
    ],
    "title": "Label",
    "type": "string"
   }
  },
  "required": [
   "label"
  ],
  "title": "ClassificationAnswer",
  "typ

## 10. Контроль качества и рассуждения для LoRA

* `judge=True`: второй проход LLM ищет ошибки разметки и ставит оценку 1–5. Примеры ниже `min_judge_score` отбрасываются.
* `reasoning=True`: после того как ответ зафиксирован и проверен, LLM пишет рассуждение, ведущее к этому ответу.
  Порядок «сначала ответ, потом рассуждение» гарантирует, что рассуждение и ответ согласованы.
* `deduplicate`: удаляет почти-дубликаты по MinHash и по косинусной близости эмбеддингов.

In [22]:
gen = sg.Generator(llm, "ner", ner_schema, judge=True, min_judge_score=3, reasoning=True)
lora_ds = await gen.run(by_topic["news"][:2] + [d for d in docs if d.id == "en-news-2"])

r = lora_ds[0]
print("оценка судьи:", r.quality["judge_score"], "| замечания:", r.quality["judge_issues"])
print("\nрассуждение:\n", r.reasoning)

ner:   0%|          | 0/3 [00:00<?, ?it/s]

оценка судьи: 3 | замечания: ["Пропущена сущность 'Татарстан' (тип LOC), так как 'Правительство Татарстана' содержит упоминание региона, но разметка не выделила 'Татарстан' отдельно", "Пропущена сущность '1 200' (тип MONEY), так как это числовое значение с количеством рабочих мест, но не денежная сумма; однако, по заданию, MONEY — только денежные суммы с валютой, поэтому это не MONEY, но пропущена сущность '1 200' как число, но по заданию не требуется выделять числа без валюты", "Неверная нормализация для 'Набережных Челнах': значение должно быть 'Набережные Челны' (именительный падеж), но в разметке указано 'Набережные Челны', что верно, однако, в тексте 'Набережных Челнах' — это родительный падеж, но по заданию value должно быть в именительном падеже, что сделано верно", "Пропущена сущность 'Корпорация' (тип ORG), так как 'корпорация «КАМАЗ»' содержит упоминание организации, но разметка выделила только 'КАМАЗ', что верно, так как по заданию нужно выделять только имя организации, без 

In [23]:
chat = export.to_chat(lora_ds, include_reasoning=True)
export.write_jsonl(chat, OUT / "sft_chat.jsonl")
for m in chat[0]["messages"]:
    print(f"--- {m['role']} ---\n{m['content'][:700]}\n")

--- system ---
Ты аккуратный аннотатор данных. Отвечай строго JSON по заданной схеме.

--- user ---
Выдели именованные сущности.
Типы сущностей:
- PER: люди
- ORG: компании, ведомства, организации
- LOC: страны, города, регионы, адреса, трассы
- DATE: даты и годы
- MONEY: денежные суммы с валютой
Поле text копируй из текста дословно, без изменения формы слов. Каждое упоминание указывай один раз. Спан — только само имя: без должностей, титулов и родовых слов (не «губернатор Иван Петров», а «Иван Петров»; не «компания «Яндекс»», а «Яндекс»).
В value укажи нормализованное значение: даты в ISO 8601, суммы числом с валютой, имена в именительном падеже.

Текст:
"""
Правительство Татарстана и корпорация «КАМАЗ» подписали соглашение о строительстве завода аккумуляторов в Набережных Челнах. Губерна

--- assistant ---
<think>
При анализе текста сначала выделяю явные организации: «Правительство Татарстана» и «КАМАЗ» как ORG, учитывая, что «корпорация» — родовое слово, его опускаю. Город «Набережн

In [24]:
# Дедупликация: добавим почти-копию одного из сгенерированных текстов
dup = gen_ds[0].model_copy(update={"id": "dup", "text": gen_ds[0].text.replace(".", "!", 1)})
unique, dedup_stats = await sg.deduplicate([*gen_ds, dup], llm=llm, embedding_threshold=0.97)
print(f"было {len(gen_ds) + 1}, осталось {len(unique)}; удалено: {dedup_stats}")

было 13, осталось 12; удалено: {'minhash': 1, 'embedding': 0}


## 11. Датасет: объединение, разбиение без утечек, чекпоинты

* `Dataset.split` делит **по исходному документу**: чанки одного документа не попадут одновременно в train и test.
* `output_path` дописывает каждую запись в JSONL сразу. Прерванный запуск на десятках тысяч документов
  продолжается с места остановки, уже сделанные примеры повторно не запрашиваются.

In [25]:
full = cls_ds + gen_ds + reg_ds + ner_ds + re_ds + qa_ds + sum_ds + real + synthetic + auto_ds + lora_ds
full.to_jsonl(OUT / "all_records.jsonl")
print(full.stats())
{k: len(v) for k, v in full.split(val=0.15, test=0.15).items()}

{'records': 80, 'by_task': {'text_classification': 22, 'text_regression': 15, 'ner': 7, 'relation_extraction': 6, 'extractive_qa': 8, 'extractive_summarization': 6, 'tabular': 16}, 'by_lang': {'ru': 65, 'en': 15}, 'by_source': {'corpus': 56, 'generated': 24}, 'schemas': 14, 'with_reasoning': 1}


{'train': 64, 'val': 8, 'test': 8}

In [26]:
ckpt = OUT / "checkpoint_demo.jsonl"
ckpt.unlink(missing_ok=True)

g1 = sg.Generator(llm, "text_classification", intent_schema, output_path=ckpt)
await g1.run(by_topic["reviews"][:2], progress=False)          # «прерванный» запуск: 2 документа из 4
calls_before = llm.usage.calls

g2 = sg.Generator(llm, "text_classification", intent_schema, output_path=ckpt)
resumed = await g2.run(by_topic["reviews"], progress=False)    # повторный запуск на всех 4
print(f"записей: {len(resumed)}, взято из чекпоинта: {g2.stats['resumed']}, новых LLM-вызовов: {llm.usage.calls - calls_before}")

записей: 5, взято из чекпоинта: 2, новых LLM-вызовов: 3


In [27]:
# Сводка по использованию LLM за весь ноутбук
llm.usage.as_dict()

{'calls': 359,
 'failed_calls': 0,
 'retries': 15,
 'repaired': 1,
 'prompt_tokens': 118344,
 'completion_tokens': 103448}

## Итог

| Файл в `output/` | Для чего |
|---|---|
| `all_records.jsonl` | канонические записи: текст, схема, ответ, уверенность, рассуждение, метаданные |
| `jev_style.jsonl` | вход, JSON-схема выхода, типизированный выход, вероятности: для модели в духе Jev |
| `sft_chat.jsonl` | messages с `<think>…</think>` для LoRA через TRL, Unsloth или axolotl |
| `ner.conll` | BIO-разметка для token classification |
| `qa_squad.json` | SQuAD v2 для экстрактивного QA |
| `flats.parquet` | типизированная таблица для CatBoost или sklearn |